### 1. Configuration

In [0]:
catalog = "dbr_dev_ua5816bd"
volume = "raw_files"
source_schema = "mialkovska_viktor594"
bronze_schema = "mialkovska_viktor594_bronze"

source_path = (
    f"/Volumes/{catalog}/{source_schema}/{volume}/"
    "smart_shipment_route_monitoring/shipment"
)

bronze_table = f"{catalog}.{bronze_schema}.shipments"


checkpoint_path = (
    f"/Volumes/{catalog}/{source_schema}/raw_files/"
    "_checkpoints/shipments"
)

schema_path = f"/Volumes/{catalog}/{source_schema}/raw_files/_schemas/shipments"


### 2. Read Source Data

Load `shipments_master.csv` with Auto Loader and add ingestion metadata.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, current_date

shipments_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .option("inferSchema", "true")
        .option("pathGlobFilter", "shipments_master.csv")
        .load(source_path)
        .withColumn("source_filename", col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

### 3. Write to Bronze

Write new records to the Bronze Delta table using a checkpoint for idempotent loading

In [0]:
query = (
    shipments_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
)

query.awaitTermination()

### 4. Validate Result

In [0]:
display(spark.table(bronze_table))